# TIAGO OLIVEIRA LIMA - 564988

In [159]:
import numpy as np

# QUESTÃO 1

## Funções auxiliares

In [160]:
# Função de normalização
def normalizar(dados):
    media = np.mean(dados, axis=0)

    matriz_de_desvios = (dados - media) ** 2

    variancia = np.sum(matriz_de_desvios, axis=0) / (dados.shape[0]-1)

    desvio_padrao = np.sqrt(variancia)

    x_normalizado = (dados - media) / desvio_padrao
    
    return x_normalizado, media, desvio_padrao

def desnormalizar(dados_normalizados, media, desvio_padrao):
    return (desvio_padrao*dados_normalizados) + media

def sigmoid(Z):
    Z = np.clip(Z, -500, 500)
    return (1/(1 + np.exp(-Z)))

def reportar_resultados_acuracia(medias, desvios, res_array, algoritmo):
    print("="*60)
    print(f"{f'RELATÓRIO DA ACURÁCIA: {algoritmo}':^60}")
    print("="*60)
    
    # Cabeçalho da Tabela de Folds
    print(f"{'Fold':<8} | {'Global':<12} | {'Classe 0':<12} | {'Classe 1':<12}")
    print("-" * 60)
    
    # Exibição de cada fold contido no res_array
    for i, fold in enumerate(res_array):
        print(f"{i+1:<8} | {fold[0]:<12.4f} | {fold[1]:<12.4f} | {fold[2]:<12.4f}")
    
    print("-" * 60)
    
    # Exibição das Estatísticas Finais (Item b da Questão 1)
    print(f"{'MÉDIA':<8} | {medias[0]:<12.4f} | {medias[1]:<12.4f} | {medias[2]:<12.4f}")
    print(f"{'DESVIO':<8} | {desvios[0]:<12.4f} | {desvios[1]:<12.4f} | {desvios[2]:<12.4f}")
    print("="*60)

## K-fold

In [ ]:
def k_fold(path, train_func, predict_func, k: int = 10, **kwargs):
    data = np.genfromtxt(path, delimiter=',', skip_header=1)
    
    np.random.seed(42)
    np.random.shuffle(data)

    folds = np.array_split(data, k)
    results = []

    for i in range(k):
        test = folds[i]
        train = np.concatenate([folds[j] for j in range(k) if j != i])
        train_norm, media, desvio_padrao = normalizar(train)
        
        X_train, y_train = train_norm[:, :-1], train_norm[:, -1:]
        X_test = (test[:, :-1] - media[:-1]) / desvio_padrao[:-1]
        y_test = test[:, -1:]
        
        weights = train_func(X_train, y_train, **kwargs)
        
        y_pred = predict_func(X_test, weights)
        
        X_test = (X_test - media[:-1]) / desvio_padrao[:-1]
        
        acc_global = np.mean(y_pred == y_test)
        acc_0 = np.mean(y_pred[y_test == 0] == y_test[y_test == 0])
        acc_1 = np.mean(y_pred[y_test == 1] == y_test[y_test == 1])
        
        results.append([acc_global, acc_0, acc_1])
                
    res_array = np.array(results)
    return np.mean(res_array, axis=0), np.std(res_array, axis=0), res_array       
                
        

## Regressão logística (GD)

In [162]:
def predict_logistic(X, w):
    probability = sigmoid(X @ w.T)
    return (probability >= 0.5).astype(int)

def train_logistic_regression(X, Y, epochs: int, alpha: float):
    N = X.shape[1]
    
    # Iniciando vetor de pesos com zeros
    w = np.zeros((1, N))
    
    # Loop de treinamento
    for i in range(epochs):
        z = X@w.T
        
        y = sigmoid(z)
        
        erro = Y - y
        
        gradiente = (X.T@erro).T/N
        w_novo = w + (alpha*gradiente)
        
        w = w_novo
    
    return w
    
    

In [163]:
medias, desvios, res_array = k_fold(
    path="breastcancer.csv", 
    train_func=train_logistic_regression, 
    predict_func=predict_logistic, 
    epochs=100, 
    alpha=0.01
)

reportar_resultados_acuracia(medias, desvios, res_array, "REGRESSÃO LOGÍSTICA")

         RELATÓRIO DA ACURÁCIA: REGRESSÃO LOGÍSTICA         
Fold     | Global       | Classe 0     | Classe 1    
------------------------------------------------------------
1        | 0.8947       | 0.9211       | 0.8421      
2        | 1.0000       | 1.0000       | 1.0000      
3        | 0.9825       | 0.9714       | 1.0000      
4        | 1.0000       | 1.0000       | 1.0000      
5        | 0.9474       | 0.9524       | 0.9333      
6        | 0.9298       | 0.9474       | 0.8947      
7        | 0.9474       | 0.9487       | 0.9444      
8        | 0.9474       | 0.9697       | 0.9167      
9        | 0.9643       | 0.9697       | 0.9565      
10       | 0.9107       | 0.8824       | 0.9545      
------------------------------------------------------------
MÉDIA    | 0.9524       | 0.9563       | 0.9442      
DESVIO   | 0.0335       | 0.0336       | 0.0483      


## GDA

In [164]:
def predict_gda(X, model):
    mu0, mu1 = model["mu0"], model["mu1"]
    sigma, pri = model["sigma"], model["pri"]
    
    # inversa da matriz de covariancia
    sigma_inv = np.linalg.inv(sigma)
    
    w1 = sigma_inv @ mu1
    b1 = -0.5 * mu1.T @ sigma_inv @ mu1 + np.log(pri)
    
    w0 = sigma_inv @ mu0
    b0 = -0.5 * mu0.T @ sigma_inv @ mu0 + np.log(1 - pri)
    
    score1 = X @ w1 + b1
    score0 = X @ w0 + b0
    
    return (score1 >= score0).astype(int).reshape(-1, 1)

def train_gda(X, y):
    n_samples, n_features = X.shape
    classes = np.unique(y)
    
    # Priori
    pri = np.mean(y == classes[1])
    
    # medias
    mu0 = np.mean(X[y.flatten() == classes[0]], axis=0)
    mu1 = np.mean(X[y.flatten() == classes[1]], axis=0)
    
    X_central = X.copy()
    X_central[y.flatten() == classes[0]] -= mu0
    X_central[y.flatten() == classes[1]] -= mu1
    
    sigma = (X_central.T @ X_central) / n_samples
    
    return {"mu0": mu0, "mu1": mu1, "sigma": sigma, "pri": pri}

In [165]:
medias_gda, desvios_gda, res_array_gda = k_fold(
    path="breastcancer.csv", 
    train_func=train_gda, 
    predict_func=predict_gda, 
    k=10 
)

reportar_resultados_acuracia(medias_gda, desvios_gda, res_array_gda, "GDA")

                 RELATÓRIO DA ACURÁCIA: GDA                 
Fold     | Global       | Classe 0     | Classe 1    
------------------------------------------------------------
1        | 0.9474       | 1.0000       | 0.8421      
2        | 0.9825       | 1.0000       | 0.9630      
3        | 1.0000       | 1.0000       | 1.0000      
4        | 0.9649       | 1.0000       | 0.9091      
5        | 0.9474       | 1.0000       | 0.8000      
6        | 0.9649       | 1.0000       | 0.8947      
7        | 0.9825       | 1.0000       | 0.9444      
8        | 0.8947       | 0.9697       | 0.7917      
9        | 0.9643       | 1.0000       | 0.9130      
10       | 0.9464       | 0.9706       | 0.9091      
------------------------------------------------------------
MÉDIA    | 0.9595       | 0.9940       | 0.8967      
DESVIO   | 0.0273       | 0.0119       | 0.0643      


## Naive Bayes Gaussiano

In [166]:
def normalizar_X(treino, teste):
    """
    Aplica Z-score apenas nos atributos, usando estatísticas do TREINO.
    """
    X_train_raw = treino[:, :-1]
    y_train = treino[:, -1:] # Mantém original
    
    X_test_raw = teste[:, :-1]
    y_test = teste[:, -1:] # Mantém original
    
    media = np.mean(X_train_raw, axis=0)
    desvio = np.std(X_train_raw, axis=0)
    desvio[desvio == 0] = 1
    
    X_train_norm = (X_train_raw - media) / desvio
    X_test_norm = (X_test_raw - media) / desvio
    
    return X_train_norm, y_train, X_test_norm, y_test

def k_fold_gnb(path, k=10):
    # Carregamento dos dados 
    data = np.genfromtxt(path, delimiter=',', skip_header=1)
    np.random.seed(42)
    np.random.shuffle(data)
    
    folds = np.array_split(data, k)
    resultados = []

    for i in range(k):
        teste_fold = folds[i]
        treino_fold = np.concatenate([folds[j] for j in range(k) if j != i])
        
        # Normalização correta (apenas X)
        X_train, y_train, X_test, y_test = normalizar_X(treino_fold, teste_fold)
        
        # Treino e Predição
        modelo = train_naive_bayes(X_train, y_train)
        y_pred = predict_naive_bayes(X_test, modelo)
        
        # Cálculo de Acurácias 
        y_test_int = y_test.astype(int).flatten()
        y_pred_int = y_pred.astype(int).flatten()
        
        acc_global = np.mean(y_pred_int == y_test_int)
        
        # Acurácia por classe
        acc_c0 = np.mean(y_pred_int[y_test_int == 0] == 0) if 0 in y_test_int else 0
        acc_c1 = np.mean(y_pred_int[y_test_int == 1] == 1) if 1 in y_test_int else 1
        
        resultados.append([acc_global, acc_c0, acc_c1])

    res = np.array(resultados)
    return np.mean(res, axis=0), np.std(res, axis=0), res

def train_naive_bayes(X, y):
    """
    Calcula os parâmetros estatísticos para cada classe.
    """
    n_samples, n_features = X.shape
    # Identifica as classes (0 e 1 no breastcancer) [cite: 11]
    classes = np.unique(y)
    
    model = {"priori": {}, "params": {}}
    
    for c in classes:
        # Filtra apenas os exemplos da classe atual
        X_c = X[y.flatten() == c]
        
        # P(y): Probabilidade a priori [cite: 14]
        model["priori"][c] = len(X_c) / n_samples
        
        # Parâmetros da Gaussiana: Média e Variância
        model["params"][c] = {
            "mu": np.mean(X_c, axis=0),
            "var": np.var(X_c, axis=0) + 1e-9  # Epsilon para estabilidade
        }
        
    return model

def predict_naive_bayes(X, model):
    """
    Classifica novos dados usando a densidade de probabilidade Gaussiana.
    """
    classes = np.array(list(model["priori"].keys()))
    log_posteriors = []

    for c in classes:
        # Log da priori para evitar underflow
        priori = np.log(model["priori"][c])
        mu = model["params"][c]["mu"]
        var = model["params"][c]["var"]
        
        # Fórmula da PDF Gaussiana no domínio do Log:
        # log(P(x|y)) = sum( -0.5 * log(2 * pi * var) - 0.5 * ((x - mu)**2 / var) )
        termo_constante = -0.5 * np.sum(np.log(2 * np.pi * var))
        termo_distancia = -0.5 * np.sum(((X - mu)**2) / var, axis=1)
        
        log_posteriors.append(priori + termo_constante + termo_distancia)
        
    # Seleciona a classe com o maior log-posterior (MAP)
    log_posteriors = np.array(log_posteriors).T
    idx_max = np.argmax(log_posteriors, axis=1)

    return classes[idx_max].reshape(-1, 1)

In [167]:
# Executando para a Questão 1
medias_nb, desvios_nb, res_array_nb = k_fold_gnb(
    path="breastcancer.csv", 
)

reportar_resultados_acuracia(medias_nb, desvios_nb, res_array_nb, "Naive Bayes Gaussiano")

        RELATÓRIO DA ACURÁCIA: Naive Bayes Gaussiano        
Fold     | Global       | Classe 0     | Classe 1    
------------------------------------------------------------
1        | 0.8947       | 0.9474       | 0.7895      
2        | 0.9649       | 1.0000       | 0.9259      
3        | 0.9825       | 0.9714       | 1.0000      
4        | 1.0000       | 1.0000       | 1.0000      
5        | 0.9649       | 0.9524       | 1.0000      
6        | 0.8772       | 0.9211       | 0.7895      
7        | 0.8947       | 0.8718       | 0.9444      
8        | 0.8947       | 1.0000       | 0.7500      
9        | 0.9286       | 0.9697       | 0.8696      
10       | 0.9286       | 0.9412       | 0.9091      
------------------------------------------------------------
MÉDIA    | 0.9331       | 0.9575       | 0.8978      
DESVIO   | 0.0406       | 0.0385       | 0.0897      


# Questão 2

In [168]:
def mapear_classes_veiculos(coluna_y):
    classes_unicas = np.unique(coluna_y)
    mapeamento = {classe: i for i, classe in enumerate(classes_unicas)}
    y_convertido = np.array([mapeamento[label] for label in coluna_y])
    return y_convertido.reshape(-1, 1), classes_unicas

def k_fold_multiclass(path, train_func, predict_func, k=10, **kwargs):
    raw_data = np.genfromtxt(path, delimiter=',', skip_header=1, dtype=str)
    
    X_raw = raw_data[:, :-1].astype(float)
    y_raw, nomes_classes = mapear_classes_veiculos(raw_data[:, -1])
    
    dataset = np.hstack((X_raw, y_raw))
    
    np.random.seed(42)
    np.random.shuffle(dataset)
    
    folds = np.array_split(dataset, k)
    results = []

    for i in range(k):
        test_fold = folds[i]
        train_fold = np.concatenate([folds[j] for j in range(k) if j != i])
        
        X_train_raw = train_fold[:, :-1]
        y_train = train_fold[:, -1:]
        
        X_test_raw = test_fold[:, :-1]
        y_test = test_fold[:, -1:].astype(int)
        
        media = np.mean(X_train_raw, axis=0)
        desvio = np.std(X_train_raw, axis=0)
        desvio[desvio == 0] = 1
        
        X_train = (X_train_raw - media) / desvio
        X_test = (X_test_raw - media) / desvio
        
        modelo = train_func(X_train, y_train, **kwargs)
        y_pred = predict_func(X_test, modelo)
        
        y_pred_flat = y_pred.flatten().astype(int)
        y_test_flat = y_test.flatten().astype(int)
        
        acc_global = np.mean(y_pred_flat == y_test_flat)
        
        acc_por_classe = []
        for c_idx in range(len(nomes_classes)):
            mask = (y_test_flat == c_idx)
            if np.any(mask):
                acc_c = np.mean(y_pred_flat[mask] == y_test_flat[mask])
            else:
                acc_c = 0.0
            acc_por_classe.append(acc_c)
            
        results.append([acc_global] + acc_por_classe)

    res_array = np.array(results)
    return np.mean(res_array, axis=0), np.std(res_array, axis=0), res_array, nomes_classes


def reportar_multiclasse(medias, desvios, res_array, nomes_classes, algoritmo):
    print("="*90)
    print(f"{f'RELATÓRIO DE ACURÁCIA MULTICLASSE: {algoritmo}':^90}")
    print("="*90)
    
    header = f"{'Fold':<8} | {'Global':<10}"
    for nome in nomes_classes:
        header += f" | {nome:<10}"
    print(header)
    print("-" * 90)
    
    for i, fold in enumerate(res_array):
        linha = f"{i+1:<8} | {fold[0]:<10.4f}"
        for j in range(len(nomes_classes)):
            linha += f" | {fold[j+1]:<10.4f}"
        print(linha)
    
    print("-" * 90)
    
    linha_media = f"{'MÉDIA':<8} | {medias[0]:<10.4f}"
    linha_desvio = f"{'DESVIO':<8} | {desvios[0]:<10.4f}"
    
    for i in range(len(nomes_classes)):
        linha_media += f" | {medias[i+1]:<10.4f}"
        linha_desvio += f" | {desvios[i+1]:<10.4f}"
        
    print(linha_media)
    print(linha_desvio)
    print("="*90)

## Regressão softmax

In [169]:
def softmax(Z):
    exp_z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def to_one_hot(y, num_classes):
    n_samples = y.size
    one_hot = np.zeros((n_samples, num_classes))
    one_hot[np.arange(n_samples), y.flatten().astype(int)] = 1
    return one_hot

def train_softmax(X, y, epochs=1000, alpha=0.01):
    X_bias = np.hstack((np.ones((X.shape[0], 1)), X))
    n_samples, n_features = X_bias.shape
    num_classes = len(np.unique(y))
    
    W = np.zeros((num_classes, n_features))
    y_encoded = to_one_hot(y, num_classes)
    
    for i in range(epochs):
        scores = X_bias @ W.T
        probabilidades = softmax(scores)
        
        erro = probabilidades - y_encoded
        gradiente = (erro.T @ X_bias) / n_samples
        W = W - (alpha * gradiente)
        
    return W

def predict_softmax(X, W):
    X_bias = np.hstack((np.ones((X.shape[0], 1)), X))
    scores = X_bias @ W.T
    probabilidades = softmax(scores)
    return np.argmax(probabilidades, axis=1).reshape(-1, 1)

# Execução no K-fold multiclasse
medias_soft, desvios_soft, res_array_soft, nomes = k_fold_multiclass(
    path="vehicle.csv", 
    train_func=train_softmax, 
    predict_func=predict_softmax, 
    k=10,
    epochs=1000,
    alpha=0.1
)

reportar_multiclasse(medias_soft, desvios_soft, res_array_soft, nomes, "REGRESSÃO SOFTMAX")

                   RELATÓRIO DE ACURÁCIA MULTICLASSE: REGRESSÃO SOFTMAX                   
Fold     | Global     | 0.000000000000000000e+00 | 1.000000000000000000e+00 | 2.000000000000000000e+00 | 3.000000000000000000e+00
------------------------------------------------------------------------------------------
1        | 0.7412     | 0.9524     | 0.5455     | 0.5769     | 1.0000    
2        | 0.7176     | 0.9048     | 0.4118     | 0.4000     | 1.0000    
3        | 0.8000     | 1.0000     | 0.7308     | 0.5652     | 1.0000    
4        | 0.7765     | 0.7647     | 0.6190     | 0.7368     | 0.9286    
5        | 0.7647     | 0.8667     | 0.5789     | 0.5789     | 1.0000    
6        | 0.7976     | 0.9310     | 0.5263     | 0.6000     | 1.0000    
7        | 0.7738     | 0.8947     | 0.6250     | 0.6429     | 0.9524    
8        | 0.7381     | 0.8667     | 0.5862     | 0.6818     | 0.9444    
9        | 0.7262     | 0.9600     | 0.4545     | 0.5714     | 0.9375    
10       | 0.8214     

## Discriminante gaussiano

In [170]:
def train_gda_multiclass(X, y):
    n_samples, n_features = X.shape
    classes = np.unique(y)
    num_classes = len(classes)
    
    phi = {}
    mu = {}
    sigma = np.zeros((n_features, n_features))
    
    for c in classes:
        X_c = X[y.flatten() == c]
        phi[c] = len(X_c) / n_samples
        mu[c] = np.mean(X_c, axis=0)
        
        diff = X_c - mu[c]
        sigma += (diff.T @ diff)
        
    sigma /= n_samples
    
    return {"mu": mu, "sigma": sigma, "phi": phi, "classes": classes}

def predict_gda_multiclass(X, model):
    mu = model["mu"]
    sigma = model["sigma"]
    phi = model["phi"]
    classes = model["classes"]
    
    sigma_inv = np.linalg.inv(sigma)
    scores = []
    
    for c in classes:
        w = sigma_inv @ mu[c]
        b = -0.5 * mu[c].T @ sigma_inv @ mu[c] + np.log(phi[c])
        score = X @ w + b
        scores.append(score)
        
    scores = np.array(scores).T
    idx_max = np.argmax(scores, axis=1)
    
    return classes[idx_max].reshape(-1, 1)

medias_gda, desvios_gda, res_array_gda, nomes = k_fold_multiclass(
    path="vehicle.csv", 
    train_func=train_gda_multiclass, 
    predict_func=predict_gda_multiclass, 
    k=10
)

reportar_multiclasse(medias_gda, desvios_gda, res_array_gda, nomes, "GDA MULTICLASSE")

                    RELATÓRIO DE ACURÁCIA MULTICLASSE: GDA MULTICLASSE                    
Fold     | Global     | 0.000000000000000000e+00 | 1.000000000000000000e+00 | 2.000000000000000000e+00 | 3.000000000000000000e+00
------------------------------------------------------------------------------------------
1        | 0.7765     | 0.9524     | 0.5909     | 0.6538     | 1.0000    
2        | 0.7647     | 0.9524     | 0.5882     | 0.4500     | 0.9630    
3        | 0.7529     | 1.0000     | 0.5769     | 0.5652     | 1.0000    
4        | 0.7765     | 0.8235     | 0.6190     | 0.7368     | 0.8929    
5        | 0.8000     | 0.9667     | 0.5789     | 0.6842     | 0.8824    
6        | 0.8095     | 0.9655     | 0.5789     | 0.5333     | 1.0000    
7        | 0.7619     | 1.0000     | 0.6250     | 0.5000     | 1.0000    
8        | 0.7976     | 0.9333     | 0.6897     | 0.6818     | 1.0000    
9        | 0.7857     | 1.0000     | 0.6364     | 0.5238     | 1.0000    
10       | 0.7976     

## Naive bays gaussiano

In [171]:
def train_naive_bayes_multiclass(X, y):
    n_samples, n_features = X.shape
    classes = np.unique(y)
    model = {"priori": {}, "params": {}}
    
    for c in classes:
        X_c = X[y.flatten() == c]
        model["priori"][c] = len(X_c) / n_samples
        model["params"][c] = {
            "mu": np.mean(X_c, axis=0),
            "var": np.var(X_c, axis=0) + 1e-9
        }
    return model

def predict_naive_bayes_multiclass(X, model):
    classes = np.array(list(model["priori"].keys()))
    posteriors = []

    for c in classes:
        priori = np.log(model["priori"][c])
        mu = model["params"][c]["mu"]
        var = model["params"][c]["var"]
        
        log_prob = -0.5 * np.sum(np.log(2 * np.pi * var)) - 0.5 * np.sum(((X - mu)**2) / var, axis=1)
        posteriors.append(priori + log_prob)
        
    posteriors = np.array(posteriors).T
    idx_max = np.argmax(posteriors, axis=1)

    return classes[idx_max].reshape(-1, 1)

medias_nb, desvios_nb, res_array_nb, nomes = k_fold_multiclass(
    path="vehicle.csv", 
    train_func=train_naive_bayes_multiclass, 
    predict_func=predict_naive_bayes_multiclass, 
    k=10
)

reportar_multiclasse(medias_nb, desvios_nb, res_array_nb, nomes, "NAIVE BAYES MULTICLASSE")

                RELATÓRIO DE ACURÁCIA MULTICLASSE: NAIVE BAYES MULTICLASSE                
Fold     | Global     | 0.000000000000000000e+00 | 1.000000000000000000e+00 | 2.000000000000000000e+00 | 3.000000000000000000e+00
------------------------------------------------------------------------------------------
1        | 0.4353     | 0.0952     | 0.4091     | 0.3846     | 1.0000    
2        | 0.4471     | 0.0476     | 0.2353     | 0.3000     | 1.0000    
3        | 0.4941     | 0.1818     | 0.6923     | 0.4348     | 0.7143    
4        | 0.5059     | 0.1765     | 0.4286     | 0.4211     | 0.8214    
5        | 0.4235     | 0.2000     | 0.4211     | 0.3684     | 0.8824    
6        | 0.3690     | 0.1724     | 0.1579     | 0.6000     | 0.6667    
7        | 0.3929     | 0.1579     | 0.4375     | 0.2143     | 0.8095    
8        | 0.4881     | 0.2000     | 0.3448     | 0.5000     | 0.9444    
9        | 0.4286     | 0.1600     | 0.5000     | 0.2381     | 1.0000    
10       | 0.4762     